# 流式输出 Streaming

流式输出让 AI 的回复像打字一样逐字显示，极大地提升了用户体验。LangChain 的 Agent 内置了完善的流式输出支持。

## 为什么需要流式输出
如果使用 invoke()，用户需要等待 Agent 完成所有步骤（多次模型调用 + 工具执行）才能看到结果。对于复杂任务，这可能耗时十几秒甚至更长。

流式输出解决了这个问题：每生成一个 Token 就立即返回，用户可以实时看到进展。
这是为您转换好的表格格式：

| 方式 | 用户体验 | 适用场景 |
| --- | --- | --- |
| invoke() | 等待 → 一次性看到完整结果 | 脚本、API、批处理 |
| stream() | 实时看到每一个 Token | 聊天界面、实时展示 |

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

print("导入完成")

导入完成


## stream_mode="messages"——逐 Token 流式

最细粒度模式，每个 chunk 对应一个 Token。

In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

model = init_chat_model("deepseek:deepseek-v4-flash")
agent = create_agent(
    model=model,
    system_prompt="你是菜鸟教程 RUNOOB 的助手。",
)

# stream_mode="messages" 逐个 Token 返回
print("实时流式输出：")
for msg_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="用一句话介绍菜鸟教程 RUNOOB")]},
    stream_mode="messages",
):
    # msg_chunk 是 AIMessageChunk
    # 每个 chunk 只包含一小段内容
    if msg_chunk.content:
        print(msg_chunk.content, end="", flush=True)
print()  # 最后换行

实时流式输出：
菜鸟教程 RUNOOB 是一个面向编程初学者的在线学习平台，提供丰富的编程语言教程、实例代码和在线工具，帮助用户快速入门和提升技能。


### 理解 metadata

metadata 包含 chunk 的来源信息。

In [5]:
print("查看 metadata 信息：\n")
for msg_chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="你好，介绍一下你自己")]},
    stream_mode="messages",
):
    if msg_chunk.content:
        print(f"内容: {msg_chunk.content}")
        print(f"来源节点: {metadata.get('langgraph_node')}")
        print(f"消息类型: {type(msg_chunk).__name__}")
        break  # 只看第一个有意义的 chunk

查看 metadata 信息：

内容: 你好
来源节点: model
消息类型: AIMessageChunk


## stream_mode="updates"——逐步查看 Agent 执行

构建显示"思考过程"的界面。

In [6]:
from langchain.tools import tool


@tool
def search_course(keyword: str) -> str:
    """在菜鸟教程 RUNOOB 搜索课程"""
    courses = {
        "python": "Python3 基础教程（30章，20小时）",
        "html": "HTML 基础教程（25章，15小时）",
    }
    return courses.get(keyword.lower(), "未找到相关课程")


agent = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[search_course],
    system_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",
)

# 使用 updates 模式查看每一步
print("=== Agent 执行过程 ===\n")
for chunk in agent.stream(
    {"messages": [HumanMessage(content="帮我查一下 Python 课程")]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"[{node_name}]", end=" ")
        if update and "messages" in update:
            for msg in update["messages"]:
                if msg.type == "ai":
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        calls = [tc['name'] for tc in msg.tool_calls]
                        print(f"请求调用: {calls}")
                    elif msg.content:
                        print(f"回复: {msg.content[:80]}")
                elif msg.type == "tool":
                    print(f"工具返回 [{msg.name}]: {msg.content}")

=== Agent 执行过程 ===

[model] 请求调用: ['search_course']
[tools] 工具返回 [search_course]: Python3 基础教程（30章，20小时）
[model] 回复: 我找到了 **Python3 基础教程**，以下是课程的详细信息：

---

## 🐍 Python3 基础教程

- **课程内容**：共 **30 章**


## stream_mode="custom"——自定义事件

Middleware 通过 `runtime.stream_writer()` 发送。

In [9]:
from langchain.agents.middleware import before_model, after_model


@before_model
def notify_before(state, runtime):
    """在模型调用前发送自定义事件"""
    runtime.stream_writer({
        "type": "status",
        "message": "正在思考...",
    })
    return None


@after_model
def notify_after(state, runtime):
    """在模型调用后发送自定义事件"""
    last_msg = state["messages"][-1] if state.get("messages") else None
    has_tools = hasattr(last_msg, 'tool_calls') and last_msg.tool_calls

    if has_tools:
        tool_names = [tc['name'] for tc in last_msg.tool_calls]
        runtime.stream_writer({
            "type": "status",
            "message": f"正在调用工具: {', '.join(tool_names)}...",
        })
    else:
        runtime.stream_writer({
            "type": "status",
            "message": "回答已完成",
        })
    return None


agent = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0),
    tools=[search_course],
    middleware=[notify_before, notify_after],
    system_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",
)

# 使用 stream_mode=["updates", "custom"] 同时接收两种事件
print("=== 混合流式输出 ===\n")
for mode, chunk in agent.stream(
    {"messages": [HumanMessage(content="查一下 Python 课程")]},
    stream_mode=["updates", "custom"],
):
    if mode == "custom":
        print(f"[自定义事件] 状态: {chunk['message']}")
    elif mode == "updates":
        for node_name, update in chunk.items():
            if update and "messages" in update:
                for msg in update["messages"]:
                    if msg.type == "ai" and msg.content:
                        print(f"[回复] {msg.content}")

=== 混合流式输出 ===

[自定义事件] 状态: 正在思考...
[回复] 好的！我来查一下 Python 课程的相关信息。
[自定义事件] 状态: 正在调用工具: search_course...
[自定义事件] 状态: 正在思考...
[回复] 找到啦！菜鸟教程 RUNOOB 为您提供 **Python3 基础教程**，以下是课程的简要信息：

## 📘 Python3 基础教程

| 项目 | 内容 |
|------|------|
| 📚 **章节数量** | **30 章** |
| ⏱ **总时长** | **约 20 小时** |
| 🎯 **适合人群** | 零基础或有一定编程基础的初学者 |

### 课程内容涵盖：
- Python3 环境搭建与基础语法
- 变量、数据类型与运算符
- 条件控制与循环语句
- 列表、元组、字典、集合等数据结构
- 函数、模块与包
- 文件操作与异常处理
- 面向对象编程
- 标准库与第三方库使用
- 等等...

### 💡 课程亮点：
- ✅ 内容通俗易懂，适合新手入门
- ✅ 每章配有代码实例，边学边练
- ✅ 在线编辑器可直接运行代码
- ✅ 完全免费学习

---

您是否想了解某个具体章节的详细内容，或者有什么其他关于 Python 学习的问题吗？比如想了解 Python 的安装、语法基础等，我都可以为您解答！😊
[自定义事件] 状态: 回答已完成


## 异步流式输出

Web 服务中使用 `astream()`。

In [12]:
import asyncio

async def stream_agent():
    agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash"), system_prompt="你是助手。")
    full = ""
    async for msg_chunk, metadata in agent.astream({"messages": [HumanMessage(content="一句话介绍菜鸟教程")]}, stream_mode="messages"):
        if msg_chunk.content:
            full += msg_chunk.content
            print(msg_chunk.content, end="", flush=True)
    print(f"\n\n长度: {len(full)} 字")

await stream_agent()
print("取消注释 await stream_agent() 运行")

菜鸟教程是一个面向编程初学者的在线学习平台，提供简洁易懂的各类技术教程与在线工具。

长度: 41 字
取消注释 await stream_agent() 运行


## FastAPI 集成示例

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

app = FastAPI()
agent = create_agent(
    model=init_chat_model("deepseek:deepseek-v4-flash"),
    system_prompt="你是菜鸟教程 RUNOOB 的助手。",
)


@app.get("/chat")
async def chat(message: str):
    """聊天接口，返回 SSE 流式响应"""
    async def generate():
        async for msg_chunk, metadata in agent.astream(
            {"messages": [HumanMessage(content=message)]},
            stream_mode="messages",
        ):
            if msg_chunk.content:
                # SSE 格式：data: xxx\n\n
                yield f"data: {msg_chunk.content}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        generate(),
        media_type="text/event-stream",
    )

# 启动：uvicorn main:app --reload

生产环境中，建议将 Agent 实例创建为全局单例，避免每次请求都重新创建。Agent 的创建开销很小（主要是编译图），但复用实例更高效。

`stream_mode` 速查表：

| 模式 | 粒度 | 迭代对象 | 典型用途 |
| --- | --- | --- | --- |
| messages | Token 级 | (AIMessageChunk, metadata) | 打字效果、实时聊天 |
| updates | 节点级 | {node_name: state_update} | 展示思考过程 |
| values | 节点级（全量） | 完整 state | 状态快照、调试 |
| custom | 自定义 | 任意 dict | 进度通知、状态推送 |
| debug | 详细 | 调试信息 | 开发阶段排查问题 |